# Imports

## Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import seaborn as sns
import math
from torch.utils.tensorboard import SummaryWriter 
import json
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from sklearn.manifold import spectral_embedding
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import rbf_kernel

## Import Dataset Classes

In [2]:
from dataset_classes import ISO_NE, ISO_NE_Small, AT, BD_Dataset, NCENT_Dataset, SH_Dataset, PL_Dataset, TN_Dataset

## Import Models

In [ ]:
from models_temporal_feature import TR_GNN_Attention, TR_GNN_GlobalLocal, TR_GNN_GraphGRU, TR_GNN_Linear, TR_GNN_MultiScale

## Import Training and Testing Loops

In [4]:
from helper_functions_trial import train_model, test_model

# Main Function

## AT

### Linear

In [ ]:
# Main Function
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    dataset = AT(
        csv_path="GLFN-TC\Datasets\AT-Dataset\AT Dataset.csv",
        T_in=72,
        T_out=240,
    )

    # --- 1. Define splits based on RAW data length ---
    total_len = len(dataset.df_numeric)
    train_split_idx = int(0.6 * total_len)
    val_split_idx = int(0.8 * total_len)
    
    print(f"Raw data length: {total_len}")
    print(f"Scaler 'train_size' (raw rows): {train_split_idx}")
    print(f"Scaler 'val_end' (raw rows): {val_split_idx}")

    # --- 2. Fit scaler only on the raw TRAIN subset ---
    scaler = StandardScaler()
    scaler.fit(dataset.df_numeric.iloc[:train_split_idx].values.astype(np.float32))

    # --- 3. Apply same scaler to all data ---
    dataset.apply_scaler(scaler)
    dataset.scaler = scaler  # keep for later inverse-transform

    # --- 4. Calculate SAMPLE indices to align with RAW splits ---
    
    # A sample 'idx' uses an input window from [idx] to [idx + T_in].
    # To prevent data leakage, the *input window* of a training sample
    # must not see any validation data. Validation data starts at `train_split_idx`.
    
    # The last training sample's input window must end at or before `train_split_idx - 1`.
    # idx + T_in - 1 <= train_split_idx - 1  =>  idx <= train_split_idx - T_in
    
    # The `range(start, end)` function's 'end' is exclusive, so this works perfectly.
    train_end = train_split_idx - dataset.T_in
    val_end = val_split_idx - dataset.T_in
    
    # Get the total number of *valid* samples
    effective_len = len(dataset)
    
    # Ensure our calculated indices don't exceed the total number of samples
    train_end = min(train_end, effective_len)
    val_end = min(val_end, effective_len)

    train_idx = range(0, train_end)
    val_idx = range(train_end, val_end)
    test_idx = range(val_end, effective_len)

    print(f"Total valid samples: {effective_len}")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
    
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    hparams = {
        "N": dataset.N,
        "T_in": 72,
        "T_out": 240,
        "d": 32,
        "hidden_dim": 64,
        "GCN_Layer": 5,
        "dropout_forecast": 0.1,
        "dropout_gcn": 0.2,
        "dropout_temporal": 0.2,
        "lr": 1e-4,
        "scheduler_patience": 3,
        "batch_size": 64, # Updated from 16
        "epochs": 100,
        "weight_decay": 1e-4, # Added this new hyperparameter
    }
    
    # --- DataLoaders ---
    train_loader = DataLoader(train_subset, batch_size=hparams["batch_size"], shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=hparams["batch_size"], shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=hparams["batch_size"], shuffle=False)
    print(f"\n🚚 DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    
    # --- Model ---
    model = TR_GNN_Linear(
        N=hparams["N"],
        T_in=hparams["T_in"],
        T_out=hparams["T_out"],
        d=hparams["d"],
        hidden_dim=hparams["hidden_dim"],
        GCN_Layer=hparams["GCN_Layer"],
        dropout_gcn=hparams["dropout_gcn"],
        dropout_temporal=hparams["dropout_temporal"],
    ).to(device)
    
    # Run name
    run = "AT_TR_GNN_Linear"
    
    log_dir = f"Final_Metrics/{run}"  # Define log directory for TensorBoard
    writer = SummaryWriter(log_dir)
    
    # Log hyperparameters as text (avoid add_hparams which requires a metric_dict)
    writer.add_text("hparams", json.dumps(hparams, indent=2))
    
    print("\n🚀 Training GLFN-TC model on AT dataset...")
    model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=hparams["epochs"],
        lr=hparams["lr"],
        device=device,
        scheduler_patience=hparams["scheduler_patience"],
        writer=writer,
        weight_decay=hparams["weight_decay"],
        save_path=f"Final_Run/{run}_best_model.pth"
    )

    print("\n🧪 Testing model performance...\n")
    preds, trues = test_model(
        dataset=dataset,
        model=model, test_loader=test_loader,
        device=device,
        writer=writer
    )
    
    writer.close()

if __name__ == "__main__":
    main()

c:\Users\khurs\Documents\GitHub\Load_Forecast_and_Balance\load_venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Loaded dataset with 9 features (target=demand), total rows=52608
Raw data length: 52608
Scaler 'train_size' (raw rows): 31564
Scaler 'val_end' (raw rows): 42086
Total valid samples: 52296
Train samples: 31492, Val samples: 10522, Test samples: 10282

🚚 DataLoaders ready. Train batches: 493, Val batches: 165, Test batches: 161

🚀 Training GLFN-TC model on AT dataset...


Epoch 1/100: 100%|██████████| 493/493 [00:06<00:00, 78.54it/s, batch_loss=0.742]


Epoch 001 | Train Loss: 1.4401 | Val Loss: 0.4751 | LR: 0.000100
✅ New best model saved (Val Loss: 0.475144)


Epoch 2/100: 100%|██████████| 493/493 [00:06<00:00, 76.14it/s, batch_loss=0.629]


...

Epoch 099 | Train Loss: 0.1219 | Val Loss: 0.0868 | LR: 0.000050
✅ New best model saved (Val Loss: 0.086760)


Epoch 100/100: 100%|██████████| 493/493 [00:05<00:00, 86.09it/s, batch_loss=0.132] 


Epoch 100 | Train Loss: 0.1220 | Val Loss: 0.0875 | LR: 0.000050
⚠️  No improvement for 1 epoch(s)

Loading best model from Final_Run/AT_GLFN_TC_Linear_best_model.pth (Val Loss: 0.086760)
Training complete. TensorBoard logs saved.

🧪 Testing model performance...



Testing: 100%|██████████| 161/161 [00:00<00:00, 255.23it/s]



Test Results:
MSE = 0.1244 | MAE = 0.2434 | R² = 0.8758

Test metrics logged to TensorBoard.


### Attention

In [ ]:
# Main Function
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    dataset = AT(
        csv_path="GLFN-TC\Datasets\AT-Dataset\AT Dataset.csv",
        T_in=72,
        T_out=240,
    )

    # --- 1. Define splits based on RAW data length ---
    total_len = len(dataset.df_numeric)
    train_split_idx = int(0.6 * total_len)
    val_split_idx = int(0.8 * total_len)
    
    print(f"Raw data length: {total_len}")
    print(f"Scaler 'train_size' (raw rows): {train_split_idx}")
    print(f"Scaler 'val_end' (raw rows): {val_split_idx}")

    # --- 2. Fit scaler only on the raw TRAIN subset ---
    scaler = StandardScaler()
    scaler.fit(dataset.df_numeric.iloc[:train_split_idx].values.astype(np.float32))

    # --- 3. Apply same scaler to all data ---
    dataset.apply_scaler(scaler)
    dataset.scaler = scaler  # keep for later inverse-transform

    # --- 4. Calculate SAMPLE indices to align with RAW splits ---
    
    # A sample 'idx' uses an input window from [idx] to [idx + T_in].
    # To prevent data leakage, the *input window* of a training sample
    # must not see any validation data. Validation data starts at `train_split_idx`.
    
    # The last training sample's input window must end at or before `train_split_idx - 1`.
    # idx + T_in - 1 <= train_split_idx - 1  =>  idx <= train_split_idx - T_in
    
    # The `range(start, end)` function's 'end' is exclusive, so this works perfectly.
    train_end = train_split_idx - dataset.T_in
    val_end = val_split_idx - dataset.T_in
    
    # Get the total number of *valid* samples
    effective_len = len(dataset)
    
    # Ensure our calculated indices don't exceed the total number of samples
    train_end = min(train_end, effective_len)
    val_end = min(val_end, effective_len)

    train_idx = range(0, train_end)
    val_idx = range(train_end, val_end)
    test_idx = range(val_end, effective_len)

    print(f"Total valid samples: {effective_len}")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
    
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    hparams = {
        "N": dataset.N,
        "T_in": 72,
        "T_out": 240,
        "d": 32,
        "hidden_dim": 64,
        "GCN_Layer": 5,
        "dropout_forecast": 0.1,
        "dropout_gcn": 0.2,
        "dropout_temporal": 0.2,
        "lr": 1e-4,
        "scheduler_patience": 3,
        "batch_size": 64, # Updated from 16
        "epochs": 100,
        "weight_decay": 1e-4, # Added this new hyperparameter
    }
    
    # --- DataLoaders ---
    train_loader = DataLoader(train_subset, batch_size=hparams["batch_size"], shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=hparams["batch_size"], shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=hparams["batch_size"], shuffle=False)
    print(f"\n🚚 DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    
    # --- Model ---
    model = TR_GNN_Attention(
        N=hparams["N"],
        T_in=hparams["T_in"],
        T_out=hparams["T_out"],
        d=hparams["d"],
        hidden_dim=hparams["hidden_dim"],
        GCN_Layer=hparams["GCN_Layer"],
        dropout_gcn=hparams["dropout_gcn"],
        dropout_temporal=hparams["dropout_temporal"],
    ).to(device)
    
    # Run name
    run = "AT_TR_GNN_Attention"
    
    log_dir = f"Final_Metrics/{run}"  # Define log directory for TensorBoard
    writer = SummaryWriter(log_dir)
    
    # Log hyperparameters as text (avoid add_hparams which requires a metric_dict)
    writer.add_text("hparams", json.dumps(hparams, indent=2))
    
    print("\n🚀 Training GLFN-TC model on AT dataset...")
    model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=hparams["epochs"],
        lr=hparams["lr"],
        device=device,
        scheduler_patience=hparams["scheduler_patience"],
        writer=writer,
        weight_decay=hparams["weight_decay"],
        save_path=f"Final_Run/{run}_best_model.pth"
    )

    print("\n🧪 Testing model performance...\n")
    preds, trues = test_model(
        dataset=dataset,
        model=model, test_loader=test_loader,
        device=device,
        writer=writer
    )
    
    writer.close()

if __name__ == "__main__":
    main()

c:\Users\khurs\Documents\GitHub\Load_Forecast_and_Balance\load_venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Loaded dataset with 9 features (target=demand), total rows=52608
Raw data length: 52608
Scaler 'train_size' (raw rows): 31564
Scaler 'val_end' (raw rows): 42086
Total valid samples: 52296
Train samples: 31492, Val samples: 10522, Test samples: 10282

🚚 DataLoaders ready. Train batches: 493, Val batches: 165, Test batches: 161

🚀 Training GLFN-TC model on AT dataset...


Epoch 1/100: 100%|██████████| 493/493 [00:05<00:00, 83.94it/s, batch_loss=0.691]


Epoch 001 | Train Loss: 6.2591 | Val Loss: 0.4568 | LR: 0.000100
✅ New best model saved (Val Loss: 0.456802)


Epoch 2/100: 100%|██████████| 493/493 [00:05<00:00, 83.69it/s, batch_loss=0.586]


...

Epoch 099 | Train Loss: 0.1235 | Val Loss: 0.0877 | LR: 0.000013
⚠️  No improvement for 2 epoch(s)


Epoch 100/100: 100%|██████████| 493/493 [00:05<00:00, 85.48it/s, batch_loss=0.157] 


Epoch 100 | Train Loss: 0.1238 | Val Loss: 0.0877 | LR: 0.000013
⚠️  No improvement for 3 epoch(s)

Loading best model from Final_Run/AT_GLFN_TC_Attention_best_model.pth (Val Loss: 0.087661)
Training complete. TensorBoard logs saved.

🧪 Testing model performance...



Testing: 100%|██████████| 161/161 [00:00<00:00, 268.38it/s]



Test Results:
MSE = 0.1227 | MAE = 0.2375 | R² = 0.8776

Test metrics logged to TensorBoard.


### GlobalLocal

In [ ]:
# Main Function
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    dataset = AT(
        csv_path="GLFN-TC\Datasets\AT-Dataset\AT Dataset.csv",
        T_in=72,
        T_out=240,
    )

    # --- 1. Define splits based on RAW data length ---
    total_len = len(dataset.df_numeric)
    train_split_idx = int(0.6 * total_len)
    val_split_idx = int(0.8 * total_len)
    
    print(f"Raw data length: {total_len}")
    print(f"Scaler 'train_size' (raw rows): {train_split_idx}")
    print(f"Scaler 'val_end' (raw rows): {val_split_idx}")

    # --- 2. Fit scaler only on the raw TRAIN subset ---
    scaler = StandardScaler()
    scaler.fit(dataset.df_numeric.iloc[:train_split_idx].values.astype(np.float32))

    # --- 3. Apply same scaler to all data ---
    dataset.apply_scaler(scaler)
    dataset.scaler = scaler  # keep for later inverse-transform

    # --- 4. Calculate SAMPLE indices to align with RAW splits ---
    
    # A sample 'idx' uses an input window from [idx] to [idx + T_in].
    # To prevent data leakage, the *input window* of a training sample
    # must not see any validation data. Validation data starts at `train_split_idx`.
    
    # The last training sample's input window must end at or before `train_split_idx - 1`.
    # idx + T_in - 1 <= train_split_idx - 1  =>  idx <= train_split_idx - T_in
    
    # The `range(start, end)` function's 'end' is exclusive, so this works perfectly.
    train_end = train_split_idx - dataset.T_in
    val_end = val_split_idx - dataset.T_in
    
    # Get the total number of *valid* samples
    effective_len = len(dataset)
    
    # Ensure our calculated indices don't exceed the total number of samples
    train_end = min(train_end, effective_len)
    val_end = min(val_end, effective_len)

    train_idx = range(0, train_end)
    val_idx = range(train_end, val_end)
    test_idx = range(val_end, effective_len)

    print(f"Total valid samples: {effective_len}")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
    
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    hparams = {
        "N": dataset.N,
        "T_in": 72,
        "T_out": 240,
        "d": 32,
        "hidden_dim": 64,
        "GCN_Layer": 5,
        "dropout_forecast": 0.1,
        "dropout_gcn": 0.2,
        "dropout_temporal": 0.2,
        "lr": 1e-4,
        "scheduler_patience": 3,
        "batch_size": 64, # Updated from 16
        "epochs": 100,
        "weight_decay": 1e-4, # Added this new hyperparameter
    }
    
    # --- DataLoaders ---
    train_loader = DataLoader(train_subset, batch_size=hparams["batch_size"], shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=hparams["batch_size"], shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=hparams["batch_size"], shuffle=False)
    print(f"\n🚚 DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    
    # --- Model ---
    model = TR_GNN_GlobalLocal(
        N=hparams["N"],
        T_in=hparams["T_in"],
        T_out=hparams["T_out"],
        d=hparams["d"],
        hidden_dim=hparams["hidden_dim"],
        GCN_Layer=hparams["GCN_Layer"],
        dropout_gcn=hparams["dropout_gcn"],
        dropout_temporal=hparams["dropout_temporal"],
    ).to(device)
    
    # Run name
    run = "AT_TR_GNN_GlobalLocal"
    log_dir = f"Final_Metrics/{run}"  # Define log directory for TensorBoard
    writer = SummaryWriter(log_dir)
    
    # Log hyperparameters as text (avoid add_hparams which requires a metric_dict)
    writer.add_text("hparams", json.dumps(hparams, indent=2))
    
    print("\n🚀 Training GLFN-TC model on AT dataset...")
    model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=hparams["epochs"],
        lr=hparams["lr"],
        device=device,
        scheduler_patience=hparams["scheduler_patience"],
        writer=writer,
        weight_decay=hparams["weight_decay"],
        save_path=f"Final_Run/{run}_best_model.pth"
    )

    print("\n🧪 Testing model performance...\n")
    preds, trues = test_model(
        dataset=dataset,
        model=model, test_loader=test_loader,
        device=device,
        writer=writer
    )
    
    writer.close()

if __name__ == "__main__":
    main()

c:\Users\khurs\Documents\GitHub\Load_Forecast_and_Balance\load_venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Loaded dataset with 9 features (target=demand), total rows=52608
Raw data length: 52608
Scaler 'train_size' (raw rows): 31564
Scaler 'val_end' (raw rows): 42086
Total valid samples: 52296
Train samples: 31492, Val samples: 10522, Test samples: 10282

🚚 DataLoaders ready. Train batches: 493, Val batches: 165, Test batches: 161

🚀 Training GLFN-TC model on AT dataset...


Epoch 1/100: 100%|██████████| 493/493 [00:06<00:00, 81.02it/s, batch_loss=0.422]


Epoch 001 | Train Loss: 1.6014 | Val Loss: 0.4344 | LR: 0.000100
✅ New best model saved (Val Loss: 0.434359)


Epoch 2/100: 100%|██████████| 493/493 [00:05<00:00, 82.84it/s, batch_loss=0.288]


...

Epoch 099 | Train Loss: 0.1097 | Val Loss: 0.0835 | LR: 0.000025
⚠️  No improvement for 4 epoch(s)


Epoch 100/100: 100%|██████████| 493/493 [00:06<00:00, 78.19it/s, batch_loss=0.139] 


Epoch 100 | Train Loss: 0.1084 | Val Loss: 0.0830 | LR: 0.000025
✅ New best model saved (Val Loss: 0.083031)

Loading best model from Final_Run/AT_GLFN_TC_GlobalLocal_best_model.pth (Val Loss: 0.083031)
Training complete. TensorBoard logs saved.

🧪 Testing model performance...



Testing: 100%|██████████| 161/161 [00:00<00:00, 250.41it/s]



Test Results:
MSE = 0.1150 | MAE = 0.2309 | R² = 0.8853

Test metrics logged to TensorBoard.


### MultiScale

In [ ]:
# Main Function
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = AT(
        csv_path="GLFN-TC\Datasets\AT-Dataset\AT Dataset.csv",
        T_in=72,
        T_out=240,
        lag_hours=[1,12,24,168], 
        rolling_windows=[12,24],
    )
    
    # 1. Prepare Data (Transpose to shape: [N_features, T_samples])
    # We want to cluster features, so each row in 'X_features' must be a feature.
    X_features = dataset.df_numeric.values.T  # Shape: (N, Total_Rows)

    # 2. Construct Affinity Matrix (RBF Kernel)
    # Gamma controls how "local" the similarity is. 
    # A lower gamma means only very close features are connected.
    # You can tune gamma (e.g., 1.0/N_features), or let sklearn handle defaults.
    affinity_matrix = rbf_kernel(X_features, gamma=None) 

    # 3. Spectral Embedding (The "Manifold" Step)
    # This maps features into a low-dimensional space (n_components) based on graph connectivity
    # Eigenvectors of the Laplacian matrix.
    embeddings = spectral_embedding(
        affinity_matrix, 
        n_components=8,  # Dimensionality of the spectral space (tuneable, usually 4-10)
        norm_laplacian=True,
        drop_first=False
    )

    # 4. Hierarchical Clustering on Embeddings (The "Grouping" Step)
    # We apply Ward's linkage on the spectral embeddings instead of raw data.
    clustering = AgglomerativeClustering(
        n_clusters=5,      # Force 5 distinct communities
        metric='euclidean', # Euclidean distance in the *Spectral Space* is valid
        linkage='ward'
    )
    cluster_labels = clustering.fit_predict(embeddings)

    # 5. Create Prior Adjacency Matrix
    N_features = len(cluster_labels)
    prior_adj_matrix = np.zeros((N_features, N_features))

    for i in range(N_features):
        for j in range(N_features):
            if cluster_labels[i] == cluster_labels[j]:
                prior_adj_matrix[i, j] = 1.0
            else:
                prior_adj_matrix[i, j] = 0.0

    # 6. Convert to Tensor
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    prior_adj_tensor = torch.tensor(prior_adj_matrix, dtype=torch.float32).to(device)
    
    # --- 1. Define splits based on RAW data length ---
    total_len = len(dataset.df_numeric)
    train_split_idx = int(0.6 * total_len)
    val_split_idx = int(0.8 * total_len)
    
    print(f"Raw data length: {total_len}")
    print(f"Scaler 'train_size' (raw rows): {train_split_idx}")
    print(f"Scaler 'val_end' (raw rows): {val_split_idx}")

    # --- 2. Fit scaler only on the raw TRAIN subset ---
    scaler = StandardScaler()
    scaler.fit(dataset.df_numeric.iloc[:train_split_idx].values.astype(np.float32))

    # --- 3. Apply same scaler to all data ---
    dataset.apply_scaler(scaler)
    dataset.scaler = scaler  # keep for later inverse-transform

    # --- 4. Calculate SAMPLE indices to align with RAW splits ---
    
    # A sample 'idx' uses an input window from [idx] to [idx + T_in].
    # To prevent data leakage, the *input window* of a training sample
    # must not see any validation data. Validation data starts at `train_split_idx`.
    
    # The last training sample's input window must end at or before `train_split_idx - 1`.
    # idx + T_in - 1 <= train_split_idx - 1  =>  idx <= train_split_idx - T_in
    
    # The `range(start, end)` function's 'end' is exclusive, so this works perfectly.
    train_end = train_split_idx - dataset.T_in
    val_end = val_split_idx - dataset.T_in
    
    # Get the total number of *valid* samples
    effective_len = len(dataset)
    
    # Ensure our calculated indices don't exceed the total number of samples
    train_end = min(train_end, effective_len)
    val_end = min(val_end, effective_len)

    train_idx = range(0, train_end)
    val_idx = range(train_end, val_end)
    test_idx = range(val_end, effective_len)

    print(f"Total valid samples: {effective_len}")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
    
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    hparams = {
        "N": dataset.N,
        "T_in": 72,
        "T_out": 240,
        "d": 32,
        "hidden_dim": 64,
        "GCN_Layer": 5,
        "dropout_forecast": 0.1,
        "dropout_gcn": 0.2,
        "dropout_temporal": 0.2,
        "lr": 1e-4,
        "scheduler_patience": 3,
        "batch_size": 64, # Updated from 16
        "epochs": 100,
        "weight_decay": 1e-4, # Added this new hyperparameter
    }
    
    # --- DataLoaders ---
    train_loader = DataLoader(train_subset, batch_size=hparams["batch_size"], shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=hparams["batch_size"], shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=hparams["batch_size"], shuffle=False)
    print(f"\n🚚 DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    
    # --- Model ---
    model = TR_GNN_MultiScale(
        N=hparams["N"],
        T_in=hparams["T_in"],
        T_out=hparams["T_out"],
        d=hparams["d"],
        hidden_dim=hparams["hidden_dim"],
        GCN_Layer=hparams["GCN_Layer"],
        dropout_gcn=hparams["dropout_gcn"],
        dropout_temporal=hparams["dropout_temporal"],
        prior_adj=prior_adj_tensor,
    ).to(device)
    
    # Run name
    run = "AT_TR_GNN_MultiScale_TR_GNN_Multiscale_Using_Temporal_Graph_Learning_and_Hierarchical_Spectral_Feature_Clustering"
    
    log_dir = f"CSE498R_Supervisor_Fixes/{run}"  # Define log directory for TensorBoard
    writer = SummaryWriter(log_dir)
    
    # Log hyperparameters as text (avoid add_hparams which requires a metric_dict)
    writer.add_text("hparams", json.dumps(hparams, indent=2))
    
    print("\n🚀 Training GLFN-TC model on AT dataset...")
    model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=hparams["epochs"],
        lr=hparams["lr"],
        device=device,
        scheduler_patience=hparams["scheduler_patience"],
        writer=writer,
        weight_decay=hparams["weight_decay"],
        save_path=f"CSE498R_Supervisor_Fixes/{run}_best_model.pth"
    )

    print("\n🧪 Testing model performance...\n")
    preds, trues = test_model(
        dataset=dataset,
        model=model, test_loader=test_loader,
        device=device,
        writer=writer
    )
    
    writer.close()

if __name__ == "__main__":
    main()

Loaded dataset with 18 features (target=demand), total rows=52440


c:\Users\khurs\Documents\GitHub\Load_Forecast_and_Balance\load_venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Raw data length: 52440
Scaler 'train_size' (raw rows): 31464
Scaler 'val_end' (raw rows): 41952
Total valid samples: 52128
Train samples: 31392, Val samples: 10488, Test samples: 10248

🚚 DataLoaders ready. Train batches: 491, Val batches: 164, Test batches: 161

🚀 Training GLFN-TC model on AT dataset...


c:\Users\khurs\Documents\GitHub\Load_Forecast_and_Balance\load_venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1/100: 100%|██████████| 491/491 [00:10<00:00, 45.86it/s, batch_loss=0.259]


Epoch 001 | Train Loss: 0.5572 | Val Loss: 0.4299 | LR: 0.000100
✅ New best model saved (Val Loss: 0.429937)


...

Epoch 099 | Train Loss: 0.0767 | Val Loss: 0.0631 | LR: 0.000013
⚠️  No improvement for 2 epoch(s)


Epoch 100/100: 100%|██████████| 491/491 [00:09<00:00, 51.96it/s, batch_loss=0.106] 


Epoch 100 | Train Loss: 0.0765 | Val Loss: 0.0630 | LR: 0.000013
⚠️  No improvement for 3 epoch(s)

               TIMING REPORT               
⏱️  Time to reach Best Model: 17m 1s
⏱️  Total Training Duration:  17m 33s

Loading best model from CSE498R_Supervisor_Fixes/AT_TR_GNN_MultiScale_TR_GNN_Multiscale_Using_Temporal_Graph_Learning_and_Hierarchical_Spectral_Feature_Clustering_best_model.pth (Val Loss: 0.062944)
Training complete. TensorBoard logs saved.

🧪 Testing model performance...



Testing: 100%|██████████| 161/161 [00:00<00:00, 171.32it/s]



Test Results:
MSE = 0.0894 | MAE = 0.2055 | R² = 0.9107

Test metrics logged to TensorBoard.
